# BƯỚC 2 CẢI TIẾN · TỐI ƯU VÀ KIỂM TRA ĐỘ BỀN
Nhóm 05 · Nhận dạng mẫu · Đề tài Voice Cloning

**Không sinh lại audio, không trích lại 6.285 clip và không cần GPU.** Notebook sử dụng cache đặc trưng từ output Bước 2 cũ, tự chọn đầu vào và chạy từ đầu đến cuối mà không cần sửa code.

## Cách chạy trên Kaggle
1. Add Input **notebook output Bước 2 cũ** chứa `buoc2_ketqua/manifest_su_dung.csv`, `dac_trung_cache.csv`, `thong_tin_thi_nghiem.json` và `bo_trich_dac_trung.py`.
2. Chọn **CPU**; bật Internet nếu thiếu thư viện và cần cài. Save Version → Save & Run All.
3. Nếu nhiều bản output khác nhau cùng được thêm, notebook dừng và yêu cầu chỉ giữ một bản, không tự chọn bừa.

## Phân biệt các phép đánh giá
- **T-B:** chọn C của SVM qua 3-fold theo ID câu trên train; so với baseline C=1, rồi tính trên test cũ.
- **T-C:** chọn SVM/RF theo CV độ bền trên train: bỏ một bộ sinh, đồng thời chia theo ID câu. Sau đó đánh giá trên test cũ và báo cáo độ bền theo bộ sinh.
- **Nhóm câu tương tự:** phép thử nhạy cảm tách nhóm bằng độ giống ký tự, **không phải test độc lập mới**.

**Quan trọng:** test cũ đã được xem ở Bước 2 và cả các thí nghiệm trước. Các kết quả này là cải tiến/thăm dò trên tập đã biết, không phải đánh giá mù cuối cùng. CV dùng để chọn mô hình cũng có thể lạc quan; muốn khẳng định tổng quát cần một tập độc lập mới. T-C chỉ là nhận dạng 7 nhãn prompt trong âm tổng hợp; không phải nhận dạng người thật chưa gặp, và không phải T-A thật/giả.

In [ ]:
# Ô 1 · MÔI TRƯỜNG VÀ CẤU HÌNH
import os,sys,glob,json,time,hashlib,shutil,subprocess,importlib,importlib.util,collections,difflib
from pathlib import Path
for var in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"):os.environ[var]="1"
def can(pkg,mod):
    if importlib.util.find_spec(mod) is not None: print("✓",pkg);return
    print("Cài",pkg,"— cần Internet")
    result=subprocess.run([sys.executable,"-m","pip","install","-q",pkg],capture_output=True,text=True)
    importlib.invalidate_caches()
    if result.returncode!=0 or importlib.util.find_spec(mod) is None:raise RuntimeError("Không cài được "+pkg+": "+(result.stderr or result.stdout)[-1200:])
for a,b in [("numpy","numpy"),("pandas","pandas"),("scikit-learn","sklearn"),("matplotlib","matplotlib"),("joblib","joblib")]:can(a,b)
import numpy as np,pandas as pd,sklearn,joblib,matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold,GroupShuffleSplit
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix,classification_report
SEED=20261103;CV=3;WORKERS=2;BOOT=2000;NGUONG_NHOM=0.80
RA=Path("/kaggle/working/buoc2_cai_tien");RA.mkdir(parents=True,exist_ok=True)
BO=["vieneu","omnivoice","omdiom"]
GIONG=["ke","u_linh","giong_nu_tre","giong_ong_gia","giong_trai_tre","giong_tre_em","tran_thy"]
CS=[0.25,1.0,3.0,10.0]
print("Thư mục kết quả:",RA,"· CPU:",WORKERS,"· sklearn:",sklearn.__version__)


In [ ]:
# Ô 2 · TỰ TÌM CACHE BƯỚC 2 VÀ KIỂM TRA ĐẦU VÀO
found=[]
for f in glob.glob("/kaggle/input/**/manifest_su_dung.csv",recursive=True):
    folder=Path(f).parent
    if (folder/"dac_trung_cache.csv").is_file() and (folder/"thong_tin_thi_nghiem.json").is_file():found.append(folder)
found=sorted(set(found))
if not found:raise FileNotFoundError("Không tìm thấy cache Bước 2. Add Input notebook output Bước 2 cũ chứa buoc2_ketqua/ và chạy lại.")
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
fp={q:tuple(sha(q/x) for x in ("manifest_su_dung.csv","dac_trung_cache.csv","thong_tin_thi_nghiem.json")) for q in found}
if len(set(fp.values()))!=1:
    raise RuntimeError("Có nhiều output Bước 2 khác nhau. Chỉ thêm đúng một output qua Add Input: "+str(found))
NGUON=found[0];print("✓ Đầu vào:",NGUON)
if len(found)>1:print("  Các bản khác có nội dung ba tệp giống nhau:",len(found))
manifest=pd.read_csv(NGUON/"manifest_su_dung.csv");feat=pd.read_csv(NGUON/"dac_trung_cache.csv")
meta=json.loads((NGUON/"thong_tin_thi_nghiem.json").read_text(encoding="utf-8"))
COLS=meta.get("feature_names",[])
can_m={"id_clip","giong","bo_sinh","cau_id","tap","noi_dung"}
if not can_m.issubset(manifest.columns) or not COLS or any(x not in feat.columns for x in COLS):raise ValueError("Thiếu cột trong manifest/cache hoặc metadata.")
if meta.get("feature_version")!="v2_f1f2f3_16k_rms_norm" or len(COLS)!=75:
    raise ValueError("Sai phiên bản hoặc số lượng đặc trưng; không trộn kết quả.")
if len(manifest)!=6285 or len(feat)!=6285 or manifest.id_clip.duplicated().any() or feat.id_clip.duplicated().any():
    raise ValueError("Manifest/cache sai số dòng hoặc có ID trùng.")
if set(manifest.id_clip)!=set(feat.id_clip):raise ValueError("ID clip trong manifest và cache khác nhau.")
D=manifest.merge(feat,on="id_clip",how="inner",validate="one_to_one")
if not np.isfinite(D[COLS].to_numpy(dtype=float)).all():raise ValueError("Có đặc trưng NaN/Inf.")
if set(D.bo_sinh)!=set(BO) or set(D.giong)!=set(GIONG) or D.tap.value_counts().to_dict()!={"train":5031,"test":1254}:
    raise ValueError("Nhãn hoặc số train/test không khớp 5031/1254.")
if set(D[D.tap=="train"].cau_id)&set(D[D.tap=="test"].cau_id):raise ValueError("Trùng ID câu train/test.")
excluded={"S0104","S0116","S0117","S0245","S0248"}
if ((D.giong=="u_linh") & D.cau_id.isin(excluded)).any():raise ValueError("15 clip lỗi chưa được lọc đúng.")
if not np.all(pd.crosstab(D.giong,D.bo_sinh).reindex(index=GIONG,columns=BO).to_numpy()==np.array([[300],[295],[300],[300],[300],[300],[300]])):
    raise ValueError("Số lượng theo bộ sinh × giọng không khớp dữ liệu đã lọc.")
train=D[D.tap=="train"].copy().reset_index(drop=True);test=D[D.tap=="test"].copy().reset_index(drop=True)
if str(meta.get("versions",{}).get("sklearn",""))!=sklearn.__version__:
    print("⚠️ sklearn khác bản cũ:",meta.get("versions",{}).get("sklearn"),"→",sklearn.__version__,"kết quả có thể sai khác nhỏ.")
print(f"✓ {len(D)} clip · {len(train)} train · {len(test)} test · {len(COLS)} đặc trưng")


In [ ]:
# Ô 3 · HÀM CHUNG, METRIC VÀ BOOTSTRAP NHÓM CÂU
MODEL_DIR=RA/"mo_hinh";MODEL_DIR.mkdir(exist_ok=True)
def make_svm(c=1.0):return Pipeline([("scale",StandardScaler()),("model",SVC(C=float(c),kernel="rbf",class_weight="balanced"))])
def make_rf():return RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight="balanced",n_jobs=WORKERS,random_state=SEED)
def score(y,p,labels):
    pr,re,f,_=precision_recall_fscore_support(y,p,labels=labels,average="macro",zero_division=0)
    return {"accuracy":float(accuracy_score(y,p)),"precision_macro":float(pr),"recall_macro":float(re),"macro_f1":float(f)}
def bootstrap_cau(y,p,groups,labels,seed=SEED):
    # Bootstrap theo câu test, không theo clip: CI có điều kiện trên model đã fit,
    # không bao gồm uncertainty do train, chọn model hay nhóm mẫu câu.
    y=np.asarray(y);p=np.asarray(p);groups=np.asarray(groups);unique=np.unique(groups)
    cms=np.stack([confusion_matrix(y[groups==g],p[groups==g],labels=labels) for g in unique])
    rng=np.random.default_rng(seed);idx=rng.integers(0,len(unique),size=(BOOT,len(unique)))
    sums=cms[idx].sum(axis=1);diag=np.diagonal(sums,axis1=1,axis2=2)
    den=sums.sum(axis=1)+sums.sum(axis=2)
    per=np.divide(2*diag,den,out=np.zeros_like(diag,dtype=float),where=den!=0)
    vals=per.mean(axis=1)
    return [float(x) for x in np.quantile(vals,[.025,.975])]
def dump_model(model,features,labels,target,name,method):
    joblib.dump({"model":model,"feature_names":features,"labels":labels,"target":target,
                 "sr":16000,"feature_version":meta["feature_version"],"selection":method},MODEL_DIR/name)
def df_out(df,name):df.to_csv(RA/name,index=False,encoding="utf-8-sig")
folds=list(GroupKFold(n_splits=CV).split(train,groups=train.cau_id))
for tr,va in folds:
    if set(train.iloc[tr].cau_id)&set(train.iloc[va].cau_id):raise RuntimeError("Rò rỉ ID câu trong CV")
print("✓ Đã chuẩn bị 3-fold theo ID câu; bootstrap theo nhóm câu trên test.")


In [ ]:
# Ô 4 · T-B: CHỌN C TRÊN TRAIN, SO VỚI BASELINE, ĐÁNH GIÁ TEST CŨ
X=train[COLS].to_numpy(dtype=float);y=train.bo_sinh.to_numpy();Xtest=test[COLS].to_numpy(dtype=float)
cv_b=[]
for C in CS:
    vals=[]
    for k,(tr,va) in enumerate(folds,1):
        model=make_svm(C);model.fit(X[tr],y[tr]); pred=model.predict(X[va])
        val=score(y[va],pred,BO)["macro_f1"];vals.append(val)
        cv_b.append({"C":C,"fold":k,"macro_f1":val,"n_train":len(tr),"n_valid":len(va)})
    print(f"C={C:g} · CV macro-F1 {np.mean(vals):.4f} ± {np.std(vals):.4f}",flush=True)
cv_b=pd.DataFrame(cv_b);df_out(cv_b,"T_B_CV_theo_fold.csv")
# Dùng std population, đồng nhất Bước 2 trước.
cv_b_sum=cv_b.groupby("C",as_index=False).agg(cv_macro_f1=("macro_f1","mean"),cv_std=("macro_f1",lambda x:float(np.std(x,ddof=0))))
cv_b_sum=cv_b_sum.sort_values(["cv_macro_f1","C"],ascending=[False,True]).reset_index(drop=True)
C_CHON=float(cv_b_sum.iloc[0].C);df_out(cv_b_sum,"T_B_chon_C_train.csv")
print("✓ C chọn trên train:",C_CHON,"— không dùng test để chọn.")
rows_b=[];b_details={};predictions=[]
for desc,C in [("baseline_C1",1.0),("chon_bang_CV",C_CHON)]:
    model=make_svm(C);model.fit(X,y);pred=model.predict(Xtest)
    values=score(test.bo_sinh,pred,BO);ci=bootstrap_cau(test.bo_sinh,pred,test.cau_id,BO)
    entry={"phien_ban":desc,"C":C,**values,"ci95_f1_thap":ci[0],"ci95_f1_cao":ci[1],"n_test":len(test)};rows_b.append(entry)
    b_details[desc]={"labels":BO,"cm":confusion_matrix(test.bo_sinh,pred,labels=BO).tolist(),"bao_cao":classification_report(test.bo_sinh,pred,labels=BO,output_dict=True,zero_division=0)}
    pp=test[["id_clip","cau_id","giong","bo_sinh"]].copy();pp["thi_nghiem"]="T_B_"+desc;pp["nhan_dung"]=test.bo_sinh;pp["du_doan"]=pred;predictions.append(pp)
    if desc=="chon_bang_CV":dump_model(model,COLS,BO,"bo_sinh","T_B_SVM_duoc_chon.joblib",f"C={C}; GroupKFold theo cau_id tren train")
    print(f"  {desc}: test cũ accuracy={values['accuracy']:.4f}, macro-F1={values['macro_f1']:.4f}; CI bootstrap theo câu [{ci[0]:.4f}, {ci[1]:.4f}]")
test_b=pd.DataFrame(rows_b);df_out(test_b,"T_B_test_cu.csv")


In [ ]:
# Ô 5 · T-C: CHỌN SVM/RF BẰNG CV CHÉO BỘ SINH TRÊN TRAIN
# Mỗi lần: bỏ 1 bộ sinh; train trên 2 bộ còn lại thuộc các câu train-fold,
# xác nhận trên bộ sinh bị bỏ thuộc các câu validation-fold.
# Bảy nhãn prompt vẫn đã xuất hiện trong training; không phải người nói mới.
cv_c=[]
for held in BO:
    for k,(ixtr,ixval) in enumerate(folds,1):
        ids_tr=set(train.iloc[ixtr].cau_id);ids_val=set(train.iloc[ixval].cau_id)
        tr=train[(train.bo_sinh!=held)&train.cau_id.isin(ids_tr)]
        va=train[(train.bo_sinh==held)&train.cau_id.isin(ids_val)]
        if set(tr.cau_id)&set(va.cau_id) or held in set(tr.bo_sinh):raise RuntimeError("Rò rỉ nhóm câu/bộ sinh")
        if set(tr.giong)!=set(GIONG) or set(va.giong)!=set(GIONG):raise RuntimeError("Fold thiếu một nhãn giọng")
        for name,creator in [("SVM",lambda:make_svm(1)),("RandomForest",make_rf)]:
            model=creator();model.fit(tr[COLS].to_numpy(dtype=float),tr.giong)
            pred=model.predict(va[COLS].to_numpy(dtype=float));metric=score(va.giong,pred,GIONG)
            cv_c.append({"bo_sinh_valid":held,"fold":k,"mo_hinh":name,"n_train":len(tr),"n_valid":len(va),**metric})
            print(f"  CV giấu {held} · fold {k} · {name}: F1={metric['macro_f1']:.4f}",flush=True)
cv_c=pd.DataFrame(cv_c);df_out(cv_c,"T_C_CV_cheo_bo_sinh_theo_fold.csv")
domain_mean=cv_c.groupby(["mo_hinh","bo_sinh_valid"],as_index=False).macro_f1.mean();df_out(domain_mean,"T_C_CV_trung_binh_moi_bo.csv")
choice=domain_mean.groupby("mo_hinh",as_index=False).agg(cv_macro_f1_trung_binh_3_bo=("macro_f1","mean")).sort_values(["cv_macro_f1_trung_binh_3_bo","mo_hinh"],ascending=[False,True]).reset_index(drop=True)
CHON_C=str(choice.iloc[0].mo_hinh);df_out(choice,"T_C_chon_mo_hinh_train.csv")
print("✓ Chọn T-C bằng CV train:",CHON_C,"\n",choice.to_string(index=False))
creator=make_rf if CHON_C=="RandomForest" else (lambda:make_svm(1))
tc_model=creator();tc_model.fit(train[COLS].to_numpy(dtype=float),train.giong);pred_c=tc_model.predict(test[COLS].to_numpy(dtype=float))
test_c=score(test.giong,pred_c,GIONG);ci_c=bootstrap_cau(test.giong,pred_c,test.cau_id,GIONG)
test_c.update(mo_hinh=CHON_C,ci95_f1_thap=ci_c[0],ci95_f1_cao=ci_c[1],n_test=len(test))
df_out(pd.DataFrame([test_c]),"T_C_test_cu_mo_hinh_chon.csv")
c_detail={"labels":GIONG,"cm":confusion_matrix(test.giong,pred_c,labels=GIONG).tolist(),"bao_cao":classification_report(test.giong,pred_c,labels=GIONG,output_dict=True,zero_division=0)}
dump_model(tc_model,COLS,GIONG,"giong","T_C_mo_hinh_duoc_chon.joblib","Trung binh macro-F1 3 bo sinh trong CV train; chia theo cau_id")
pp=test[["id_clip","cau_id","giong","bo_sinh"]].copy();pp["thi_nghiem"]="T_C_mo_hinh_chon";pp["nhan_dung"]=test.giong;pp["du_doan"]=pred_c;predictions.append(pp)
print(f"T-C {CHON_C}: test cũ accuracy={test_c['accuracy']:.4f} · macro-F1={test_c['macro_f1']:.4f}")

# Bảng test theo bộ sinh đã được xem trước: chỉ là thăm dò so sánh, không dùng chọn model.
lodo=[];pred_lodo=[]
for held in BO:
    tr=train[train.bo_sinh!=held];te=test[test.bo_sinh==held]
    for name,creator in [("SVM",lambda:make_svm(1)),("RandomForest",make_rf)]:
        model=creator();model.fit(tr[COLS].to_numpy(dtype=float),tr.giong);pred=model.predict(te[COLS].to_numpy(dtype=float))
        met=score(te.giong,pred,GIONG);lodo.append({"bo_sinh_test":held,"mo_hinh":name,"n_train":len(tr),"n_test":len(te),**met})
        pp=te[["id_clip","cau_id","giong","bo_sinh"]].copy();pp["bo_sinh_giu_lai"]=held;pp["mo_hinh"]=name;pp["nhan_dung"]=te.giong;pp["du_doan"]=pred;pred_lodo.append(pp)
        print(f"  Thăm dò test cũ — giữ {held}, {name}: macro-F1={met['macro_f1']:.4f}",flush=True)
lodo=pd.DataFrame(lodo);df_out(lodo,"T_C_tham_do_test_cheo_bo_sinh.csv")
df_out(pd.concat(pred_lodo,ignore_index=True),"T_C_du_doan_test_cheo_bo_sinh.csv")


In [ ]:
# Ô 6 · PHÉP THỬ NHẠY CẢM: TÁCH NHÓM CÂU TƯƠNG TỰ
# Nhóm nối thành phần liên thông nếu similarity ký tự >= 0,80. Đây là heuristic,
# không khẳng định nhóm ngữ nghĩa hoàn hảo; chọn cố định trước khi tính điểm.
text_counts=D.groupby("cau_id").noi_dung.nunique()
if (text_counts!=1).any():raise ValueError("Một câu ID có nhiều nội dung khác nhau")
txt=D[["cau_id","noi_dung"]].drop_duplicates().sort_values("cau_id").reset_index(drop=True)
parent=list(range(len(txt)))
def find(i):
    while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
    return i
def union(a,b):
    a,b=find(a),find(b)
    if a!=b:parent[a]=b
ss=txt.noi_dung.fillna("").str.lower().tolist()
for i in range(len(ss)):
    for j in range(i):
        if difflib.SequenceMatcher(None,ss[i],ss[j]).ratio()>=NGUONG_NHOM:union(i,j)
txt["nhom_mau_cau"]=[f"G{find(i):03d}" for i in range(len(txt))]
if txt.nhom_mau_cau.nunique()<5:raise ValueError("Quá ít nhóm mẫu câu để tách")
# Chia theo nhóm, không theo clip; ánh xạ lại toàn bộ các giọng/bộ sinh.
tr_i,te_i=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(txt,groups=txt.nhom_mau_cau))
txt["tap_nhom_tham_do"]="train";txt.loc[te_i,"tap_nhom_tham_do"]="test"
if set(txt.iloc[tr_i].nhom_mau_cau)&set(txt.iloc[te_i].nhom_mau_cau):raise RuntimeError("Trùng nhóm mẫu câu")
df_out(txt,"bang_nhom_mau_cau_tham_do.csv")
templ=D.merge(txt[["cau_id","nhom_mau_cau","tap_nhom_tham_do"]],on="cau_id",validate="many_to_one")
tr=templ[templ.tap_nhom_tham_do=="train"];te=templ[templ.tap_nhom_tham_do=="test"]
print(f"{txt.nhom_mau_cau.nunique()} nhóm; {len(tr)} clip train / {len(te)} clip test theo nhóm câu gần nhau")
print("Tập chia mới có thể chứa clip từng nằm trong test cũ, nên không phải tập test độc lập.")
template_scores=[]
for name,target,labels,creator in [("T_B_SVM_C1","bo_sinh",BO,lambda:make_svm(1)),("T_C_RF_co_dinh","giong",GIONG,make_rf)]:
    if set(tr[target])!=set(labels) or set(te[target])!=set(labels):raise ValueError("Tập nhóm câu thiếu nhãn "+target)
    model=creator();model.fit(tr[COLS].to_numpy(dtype=float),tr[target]);pred=model.predict(te[COLS].to_numpy(dtype=float))
    met=score(te[target],pred,labels);template_scores.append({"thi_nghiem":name,"n_train":len(tr),"n_test":len(te),"so_nhom_train":txt.iloc[tr_i].nhom_mau_cau.nunique(),"so_nhom_test":txt.iloc[te_i].nhom_mau_cau.nunique(),**met})
    print(f"  {name}: accuracy={met['accuracy']:.4f}, macro-F1={met['macro_f1']:.4f}")
template_scores=pd.DataFrame(template_scores);df_out(template_scores,"tham_do_tach_nhom_mau_cau.csv")


In [ ]:
# Ô 7 · DỰNG HTML TỰ CHỨA VÀ HÌNH PNG CHO SLIDE
import base64,io,html
plt.rcParams.update({"figure.facecolor":"white","axes.spines.top":False,"axes.spines.right":False,"font.size":10.5,"axes.titlesize":12,"axes.titleweight":"bold","figure.autolayout":True})
HINH=[];figdir=RA/"hinh";figdir.mkdir(exist_ok=True)
def savefig(title,note,fig,slug):
    buf=io.BytesIO();fig.savefig(buf,format="png",dpi=150,bbox_inches="tight",facecolor="white");plt.close(fig)
    raw=buf.getvalue();(figdir/(slug+".png")).write_bytes(raw)
    HINH.append((title,note,base64.b64encode(raw).decode()))
def plotcm(cm,labels,title,slug):
    cm=np.asarray(cm);norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    fig,ax=plt.subplots(figsize=(max(6,len(labels)*1.0),max(4,len(labels)*.7)));im=ax.imshow(norm,cmap="Blues",vmin=0,vmax=1)
    ax.set_xticks(range(len(labels)),labels,rotation=32,ha="right");ax.set_yticks(range(len(labels)),labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j,i,f"{cm[i,j]}\n{norm[i,j]*100:.0f}%",ha="center",va="center",fontsize=8,color="white" if norm[i,j]>.52 else "#17232d")
    ax.set_xlabel("Dự đoán");ax.set_ylabel("Nhãn đúng");ax.set_title(title);ax.grid(False);fig.colorbar(im,ax=ax,shrink=.72,label="tỷ lệ trong hàng")
    savefig(title,"Số ô là số clip, tỷ lệ tính theo hàng. Kết quả trên test cũ đã từng được xem.",fig,slug)

fig,ax=plt.subplots(figsize=(8,3.6));v=cv_b_sum.sort_values("C");x=np.arange(len(v));ax.bar(x,v.cv_macro_f1,color=["#21849e" if c==C_CHON else "#9cbac7" for c in v.C],width=.6,yerr=v.cv_std,capsize=4)
ax.set_xticks(x,[f"C={x:g}" for x in v.C]);ax.set_ylim(0,1.08);ax.set_ylabel("macro-F1 CV");ax.set_title("T-B · Chọn C chỉ trên train")
for i,a in enumerate(v.cv_macro_f1):ax.text(i,a+.04,f"{a:.3f}",ha="center",weight="bold")
savefig("T-B — chọn C trên train","3-fold theo câu. Thanh sai số là độ lệch chuẩn 3 fold, không phải khoảng tin cậy. Giá trị tốt nhất có thể lạc quan do đã chọn tham số.",fig,"01_chon_C")
fig,ax=plt.subplots(figsize=(7,3.4));ax.bar([0,1],test_b.macro_f1,color=["#9cbac7","#21849e"],width=.58);ax.set_xticks([0,1],["Baseline C=1",f"Chọn CV C={C_CHON:g}"]);ax.set_ylim(0,1.08);ax.set_ylabel("macro-F1 test cũ");ax.set_title("T-B · So với baseline")
for i,v in enumerate(test_b.macro_f1):ax.text(i,v+.025,f"{v:.3f}",ha="center",weight="bold")
savefig("T-B — kết quả trên test cũ","Test này đã được xem trước, vì vậy đây không phải khẳng định trên tập mù mới.",fig,"02_TB_test")
plotcm(b_details["chon_bang_CV"]["cm"],BO,"T-B · SVM được chọn · test cũ","03_TB_confusion")
fig,ax=plt.subplots(figsize=(9,3.6));pivot=domain_mean.pivot(index="bo_sinh_valid",columns="mo_hinh",values="macro_f1").reindex(BO);xx=np.arange(3)
for offset,name,col in [(-.18,"SVM","#e48751"),(.18,"RandomForest","#21849e")]:ax.bar(xx+offset,pivot[name],width=.34,label=name,color=col)
ax.set_xticks(xx,BO);ax.set_ylim(0,1.08);ax.set_ylabel("macro-F1 CV train");ax.set_title("T-C · CV bỏ một bộ sinh trong train");ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.01,.5))
savefig("T-C — CV độ bền trên train","Mỗi điểm trung bình trên 3 fold; bộ sinh xác nhận không có trong train fold và câu cũng khác ID. Cùng bảy nhãn prompt vẫn xuất hiện ở cả hai phía.",fig,"04_TC_cv_domain")
fig,ax=plt.subplots(figsize=(9,3.6));p=lodo.pivot(index="bo_sinh_test",columns="mo_hinh",values="macro_f1").reindex(BO)
for offset,name,col in [(-.18,"SVM","#e48751"),(.18,"RandomForest","#21849e")]:
    ax.bar(xx+offset,p[name],width=.34,label=name,color=col)
    for i,vv in enumerate(p[name]):ax.text(i+offset,vv+.025,f"{vv:.3f}",ha="center",fontsize=9)
ax.set_xticks(xx,BO);ax.set_ylim(0,1.1);ax.set_ylabel("macro-F1 test cũ");ax.set_title("T-C · Thăm dò bộ sinh chưa gặp");ax.legend(frameon=False,loc="center left",bbox_to_anchor=(1.01,.5))
savefig("T-C — test chéo bộ sinh đã xem","So sánh bổ sung sau khi test đã xem; không dùng bảng test này để chọn mô hình. Hai bộ sinh còn lại nằm trong train.",fig,"05_TC_test_domain")
plotcm(c_detail["cm"],GIONG,"T-C · "+CHON_C+" được chọn · test cũ","06_TC_confusion")
fig,ax=plt.subplots(figsize=(8,3.5));xx=np.arange(len(template_scores));ax.bar(xx,template_scores.macro_f1,color=["#21849e","#9b70b6"],width=.55)
ax.set_xticks(xx,template_scores.thi_nghiem);ax.set_ylim(0,1.08);ax.set_ylabel("macro-F1");ax.set_title("Phép thử nhạy cảm theo nhóm câu")
for i,vv in enumerate(template_scores.macro_f1):ax.text(i,vv+.025,f"{vv:.3f}",ha="center",weight="bold")
savefig("Tách nhóm câu tương tự — thăm dò","Nhóm theo tương đồng ký tự, ngưỡng 0,80. Tập được chia lại trên toàn bộ dữ liệu đã sử dụng; không phải tập độc lập mới.",fig,"07_nhom_cau")

# Ghi các bảng kết quả và dự đoán.
df_out(pd.concat(predictions,ignore_index=True),"du_doan_test_cu.csv")
with open(RA/"ma_tran_va_chi_tiet.json","w",encoding="utf-8") as f:json.dump({"T_B":b_details,"T_C":c_detail},f,ensure_ascii=False,indent=2)
def table(df):return df.to_html(index=False,border=0,classes="table",float_format=lambda x:f"{x:.4f}",escape=True)
css="""body{margin:0;background:#f3f7fa;color:#182a39;font:16px/1.58 system-ui,-apple-system,Segoe UI,sans-serif}header{background:linear-gradient(125deg,#102f44,#197893);color:white;padding:40px 24px}header div,main{max-width:1100px;margin:auto}main{padding:20px 18px 65px}h1{margin:0}h2{margin:38px 0 12px;color:#17647d;border-bottom:2px solid #dae6eb}h3{margin-top:8px}figure,.box{background:white;border:1px solid #dce6ed;border-radius:12px;padding:18px;margin:18px 0;overflow:auto}figure img{max-width:100%;height:auto;display:block;margin:auto}figcaption{color:#4d6471;border-left:3px solid #1f8da3;padding-left:12px;font-size:14px}table.table{border-collapse:collapse;width:100%;font-size:13px;white-space:nowrap}table.table th,table.table td{padding:8px;border-bottom:1px solid #e4eaee;text-align:left}table.table th{background:#e8f2f7}.warn{background:#fff7e9;border-left:4px solid #e8a43b;padding:14px 16px}code{background:#eef3f6;border-radius:3px;padding:2px 4px}@media print{body{background:#fff}figure{break-inside:avoid}}"""
parts=["<!doctype html><html lang='vi'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>Bước 2 cải tiến</title><style>",css,"</style></head><body><header><div><h1>Bước 2 · Tối ưu &amp; kiểm tra độ bền</h1><p>Nhóm 05 · Đề tài Voice Cloning · 6.285 clip · 75 đặc trưng</p></div></header><main>"]
parts.append("<div class='warn'><b>Giới hạn quan trọng:</b> Tập test cũ đã được xem ở Bước 2 và các phân tích trước, nên không phải đánh giá mù mới. Phép chia nhóm câu là phân tích nhạy cảm trên cùng corpus, không phải dữ liệu độc lập. Bảy prompt/nhãn giọng đã xuất hiện trong train; chưa chứng minh tổng quát sang người thật hoặc giọng chưa gặp. Đây không phải bài toán thật/giả.</div>")
parts.append("<h2>1. T-B — chọn SVM trên train</h2><div class='box'><p>Chọn C theo macro-F1 GroupKFold 3 theo ID câu trên train. CV của cấu hình thắng có thể lạc quan do chọn tham số. Test cũ chỉ để đối chiếu.</p>"+table(cv_b_sum)+"<h3>Đối chiếu test cũ</h3>"+table(test_b)+"</div>")
parts.append("<h2>2. T-C — chọn mô hình theo độ bền trên train</h2><div class='box'><p>CV: giữ lại một bộ sinh làm validation, đồng thời tách theo ID câu; lấy trung bình macro-F1 trên 3 bộ sinh. Dùng cách này chọn mô hình trước khi tính các bảng test của notebook.</p>"+table(choice)+"<h3>Trung bình mỗi bộ sinh trong CV train</h3>"+table(domain_mean)+"<h3>Test cũ của mô hình được chọn</h3>"+table(pd.DataFrame([test_c]))+"<h3>Thăm dò test cũ — bộ sinh chưa gặp</h3>"+table(lodo)+"</div>")
parts.append("<h2>3. Phép thử nhạy cảm theo nhóm câu</h2><div class='box'><p>Độ giống ký tự SequenceMatcher ≥ 0,80 tạo cạnh, sau đó lấy thành phần liên thông; tách 20% số nhóm với seed cố định. Đây là heuristic và dữ liệu chia lại trên cùng corpus đã xem, không phải tập độc lập. Mô hình/hyperparameter cố định để mô tả, không dùng bảng này để chọn mô hình.</p>"+table(template_scores)+"</div>")
parts.append("<h2>4. Hình trực quan</h2>")
for title,note,b64 in HINH:parts.append("<figure><h3>"+html.escape(title)+"</h3><img src='data:image/png;base64,"+b64+"'><figcaption>"+html.escape(note)+"</figcaption></figure>")
parts.append("<h2>5. Cách hiểu khoảng tin cậy</h2><div class='box'><p>CI 95% dùng 2.000 mẫu bootstrap theo <b>ID câu</b>, giữ nguyên mô hình đã huấn luyện. CI không bao gồm độ bất định do huấn luyện/chọn mô hình, rò rỉ các mẫu câu tương tự, hay thiếu giọng và bộ sinh mới. Giá trị độ lệch chuẩn CV không phải CI.</p><p>Đặc trưng được lấy từ cache Bước 2: 16 kHz, F1 MFCC+delta, F2 phổ, F3 nhịp/thời lượng. Khác biệt codec, điều kiện giọng, hoặc tiền xử lý vẫn có thể ảnh hưởng kết quả sau resample.</p></div></main></body></html>")
report=RA/"BAO_CAO_BUOC2_CAI_TIEN.html";report.write_text("".join(parts),encoding="utf-8")
print(f"✅ HTML: {report} · {len(HINH)} hình")


In [ ]:
# Ô 8 · LƯU DẤU VẾT, BỘ TRÍCH VÀ ZIP
old_extractor=NGUON/"bo_trich_dac_trung.py"
if old_extractor.is_file():
    shutil.copy2(old_extractor,RA/"bo_trich_dac_trung.py")
    print("✓ Đã kèm bộ trích đặc trưng gốc Bước 2")
else:print("⚠️ Output Bước 2 không có bo_trich_dac_trung.py; cần notebook Bước 2 gốc để suy luận audio.")
guide="""# Dùng lại mô hình

Tệp joblib phải được tạo và nạp trong môi trường đáng tin cậy, cùng phiên bản scikit-learn nếu có thể. Mã đặc trưng được sao chép từ output Bước 2.

```python
import numpy as np
import soundfile as sf
import joblib
from bo_trich_dac_trung import dac_trung_mot
p = 'duong_dan_clip.flac'  # thay bằng audio thực tế
r = {'id_clip':'clip_moi', 'duong_dan_input':p, 'sr':sf.info(p).samplerate}
f, loi = dac_trung_mot(r)
if loi: raise RuntimeError(loi)
b = joblib.load('mo_hinh/T_B_SVM_duoc_chon.joblib')
x = np.array([[f[k] for k in b['feature_names']]], dtype=float)
print(b['model'].predict(x)[0])
```

T-B chỉ dự đoán ba bộ sinh đã huấn luyện; T-C dự đoán bảy nhãn prompt đã có. Không dùng để kết luận thật/giả hoặc danh tính người thật mới.
"""
(RA/"HUONG_DAN_MO_HINH.md").write_text(guide,encoding="utf-8")
metadata={"nguon":str(NGUON),"input_sha256":list(fp[NGUON]),"seed":SEED,"cv_folds":CV,"n_train":len(train),"n_test":len(test),"feature_names":COLS,"feature_version":meta['feature_version'],"C_da_chon":C_CHON,"T_C_duoc_chon":CHON_C,"template_similarity":NGUONG_NHOM,"bootstrap_repeats":BOOT,"python":sys.version,"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__,"ghi_chu":"Test cu da xem; split nhom cau tham do; khong phai danh gia doc lap moi."}
(RA/"thong_tin_cai_tien.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
zip_path=shutil.make_archive("/kaggle/working/KET_QUA_BUOC2_CAI_TIEN_"+time.strftime("%m%d_%H%M%S"),"zip",RA)
print("\n📄 HTML:",report)
print(f"📦 ZIP: {zip_path} · {os.path.getsize(zip_path)/1024**2:.1f} MiB")
print("Gửi HTML + T_B_test_cu.csv + T_C_CV_trung_binh_moi_bo.csv + T_C_tham_do_test_cheo_bo_sinh.csv + tham_do_tach_nhom_mau_cau.csv.")
print("Kết quả test đã xem chỉ là đánh giá bổ sung; không gọi là test mù mới.")
